# Day 22 — New Tool
### #30DayChartChallenge | April 2026

**The Relentless Rise: 3 Decades of Global Mean Sea Level Change.**

**New Tool:** `gganimate`

**Data:** [NOAA Laboratory for Satellite Altimetry](https://www.star.nesdis.noaa.gov/socd/lsa/SeaLevelRise/)  
**Author:** Sharfudeen Yasar Arafath

In [1]:
library(ggplot2)
library(dplyr)
library(tidyr)
library(showtext)
library(sysfonts)
library(scales)
library(gganimate)
library(gifski)


Attaching package: 'dplyr'


The following objects are masked from 'package:stats':

    filter, lag


The following objects are masked from 'package:base':

    intersect, setdiff, setequal, union


Loading required package: sysfonts

Loading required package: showtextdb



In [2]:
font_add_google("Outfit", "outfit")
font_add_google("Roboto Condensed", "roboto_condensed")
font_add_google("JetBrains Mono", "jetbrains")
showtext_auto()
showtext_opts(dpi = 150)

options(repr.plot.width = 13, repr.plot.height = 8, repr.plot.res = 150)

In [3]:
bg_col   <- "#0a0e17"
text_col <- "#e0e6f0"
grid_col <- "#1a2030"

mission_colours <- c(
  "TOPEX/Poseidon" = "#ef476f",
  "Jason-1"        = "#ffd166",
  "Jason-2"        = "#06d6a0",
  "Jason-3"        = "#118ab2",
  "Sentinel-6MF"   = "#e0e6f0"
)

In [4]:
raw <- read.csv("../../data/day_22/slr_sla_gbl_keep_ref_90.csv",
                comment.char = "#", stringsAsFactors = FALSE)

df <- raw %>%
  rename(`TOPEX/Poseidon` = TOPEX.Poseidon,
         `Jason-1` = Jason.1, `Jason-2` = Jason.2,
         `Jason-3` = Jason.3, `Sentinel-6MF` = Sentinel.6MF) %>%
  pivot_longer(-year, names_to = "mission", values_to = "sla_mm") %>%
  filter(!is.na(sla_mm)) %>%
  arrange(year) %>%
  mutate(mission = factor(mission,
    levels = c("TOPEX/Poseidon","Jason-1","Jason-2","Jason-3","Sentinel-6MF")))

cat("Rows:", nrow(df), "| Year:", range(df$year), "\n")

Rows: 1657 | Year: 1992.961 2025.127 


In [5]:
chart_theme <- theme_minimal(base_family = "outfit", base_size = 14) +
  theme(
    plot.background    = element_rect(fill = bg_col, colour = NA),
    panel.background   = element_rect(fill = bg_col, colour = NA),
    panel.grid.major.x = element_blank(),
    panel.grid.minor   = element_blank(),
    panel.grid.major.y = element_line(colour = grid_col, linewidth = 0.3),
    plot.title    = element_text(family = "outfit", face = "bold",
                                size = 24, colour = text_col,
                                margin = margin(b = 4)),
    plot.subtitle = element_text(family = "roboto_condensed",
                                size = 13, colour = alpha(text_col, 0.7),
                                margin = margin(b = 12)),
    plot.caption  = element_text(family = "jetbrains", size = 8,
                                colour = alpha(text_col, 0.5),
                                hjust = 1, lineheight = 1.4,
                                margin = margin(t = 20)),
    axis.text     = element_text(family = "jetbrains", size = 9,
                                colour = alpha(text_col, 0.6)),
    axis.title.y  = element_text(family = "roboto_condensed", size = 11,
                                colour = alpha(text_col, 0.7),
                                margin = margin(r = 8)),
    legend.position      = "top",
    legend.justification = "left",
    legend.text   = element_text(family = "roboto_condensed", size = 10,
                                colour = alpha(text_col, 0.8)),
    legend.title  = element_text(family = "roboto_condensed", size = 11,
                                colour = text_col, face = "bold"),
    legend.key    = element_rect(fill = NA, colour = NA),
    plot.margin   = margin(20, 24, 24, 14)
  )

chart_labels <- labs(
  title    = "The Relentless Rise",
  subtitle = "Global Mean Sea Level Anomaly, 1993\u20132025 (mm) \u2014 Satellite Altimetry",
  caption  = "#30DayChartChallenge \u00b7 Day 22 \u00b7 New Tool (gganimate)\nData: NOAA Laboratory for Satellite Altimetry \u00b7 Trend: +3.1 mm/year\nAuthor: Sharfudeen Yasar Arafath",
  x = NULL,
  y = "Sea\u2011level anomaly (mm)"
)

chart_scales <- list(
  scale_colour_manual(values = mission_colours, name = "Mission"),
  scale_x_continuous(breaks = seq(1993, 2025, 4),
                     expand = expansion(mult = c(0.02, 0.05))),
  scale_y_continuous(labels = function(x) paste0(ifelse(x > 0, "+", ""), x),
                     breaks = seq(-30, 100, 10),
                     expand = expansion(mult = c(0.05, 0.08)))
)

In [6]:
p_anim <- ggplot(df, aes(x = year, y = sla_mm, colour = mission)) +
  geom_hline(yintercept = 0, colour = grid_col, linewidth = 0.4) +
  geom_line(linewidth = 0.9, alpha = 0.9) +
  geom_point(size = 2.5) +
  chart_scales + chart_labels + chart_theme +
  transition_reveal(year) +
  ease_aes("linear")

anim <- suppressWarnings(animate(
  p_anim,
  nframes   = 150,
  fps       = 15,
  width     = 1600,
  height    = 900,
  res       = 150,
  renderer  = gifski_renderer(),
  end_pause = 20
))

anim_save("../../chart/day_22_new_tool.gif", animation = anim)
cat("\u2705 GIF saved\n")

✅ GIF saved


In [7]:
p_static <- ggplot(df, aes(x = year, y = sla_mm, colour = mission)) +
  geom_hline(yintercept = 0, colour = grid_col, linewidth = 0.4) +
  geom_line(linewidth = 0.9, alpha = 0.9) +
  chart_scales + chart_labels + chart_theme

showtext_opts(dpi = 300)
ggsave("../../chart/day_22_new_tool.png", plot = p_static,
       width = 13, height = 8, dpi = 300, bg = bg_col)
cat("\u2705 PNG saved\n")

✅ PNG saved
